Robustness check with suggestive misinformation included in accurate information category 

In [1]:
# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score


from sklearn.model_selection import train_test_split

import emoji
import re
import datasets
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, RobertaForSequenceClassification
import warnings

c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data

In [ ]:
data =  pd.read_csv("Data_rob_sug.csv")

Get tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

Get BERT model

In [37]:
model = AutoModelForSequenceClassification.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at DTAI-KULeuven/robbert-2022-dutch-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Prepare data

In [7]:
# Classification only works when outcome variable is called labels
data[['label']] = data[['mis_robust']]
# Convert emoji to descriptions (in English) using emoji package
def no_emoji(text):
    text = emoji.demojize(text) 
    return text 
data['text'] = data['text'].apply(no_emoji)
#Remove urls, &gt, &lt and &amp, and [numbers] from text
#Removed [numbers] because there were meaningless numbers between brackets in the Tweets
def clean_text(text):
    text = re.sub(r'https?://\S+|www\.\S+|\r|\n|&gt.?| &lt.?|&amp.?|\[\d*\]', '', text)
    return text 
data['text'] = data['text'].apply(clean_text) 

#Display cleaned text
pd.set_option('display.max_rows', None)
pd.set_option('max_colwidth', None)

#One dataset with only text and labels to train BERT model
#One dataset with also year so I can later split dataset and evaluate metrics per year
data_inf = data[['text', 'label', 'id', 'year']]
data = data[['text', 'label']]



Set values of variables important for training

In [ ]:
max_length = 512
# set random seed for reproducibility
SEED_GLOBAL = 42
np.random.seed(SEED_GLOBAL)
#Set training directory 
training_directory = ""

Split data into training and test set

In [8]:
# Train and test set for training model
df_train, df_test = train_test_split(data, random_state=42, test_size=0.25)

#Create identical test with text and labels plus year so performance metrics can be split by year
df_train_inf, df_test_inf = train_test_split(data_inf, random_state=42, test_size=0.25)

Tokenize data (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [9]:
# convert pandas dataframes to Hugging Face dataset object to facilitate pre-processing
dataset = datasets.DatasetDict({
    "train": datasets.Dataset.from_pandas(df_train),
    "test": datasets.Dataset.from_pandas(df_test)
})

# tokenize
def tokenize(examples):
  return tokenizer(examples["text"], truncation=True, padding = 'max_length', max_length=512)  # max_length can be reduced to e.g. 256 to increase speed, but long texts will be cut off

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 849/849 [00:00<00:00, 3685.62 examples/s]


Set training arguments (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [43]:
train_args = TrainingArguments(
    num_train_epochs=3,  
    learning_rate=2e-5,
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=64,   
    warmup_ratio=0.06, 
    weight_decay=0.1,
    seed=SEED_GLOBAL,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    evaluation_strategy="epoch", 
    save_strategy = "epoch",
    report_to="all",
    output_dir=f'{training_directory}',
    logging_dir=f'{training_directory}',
)


Set evaluation metrics (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [ ]:
# Function to calculate metrics
# documentation on all metrics: https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics

def compute_metrics_standard(eval_pred):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        labels = eval_pred.label_ids
        pred_logits = eval_pred.predictions
        preds_max = np.argmax(pred_logits, axis=1) 

        # metrics
        precision_mis, recall_mis, f1_mis, _ = precision_recall_fscore_support(labels, preds_max, average= 'binary') 
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds_max, average='macro')  
        precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds_max, average='micro')  
        acc_balanced = balanced_accuracy_score(labels, preds_max)
        acc_not_balanced = accuracy_score(labels, preds_max)

        metrics = {
            'accuracy': acc_not_balanced,
            'f1_macro': f1_macro,
            'accuracy_balanced': acc_balanced,
            'f1_micro': f1_micro,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'precision_micro': precision_micro,
            'recall_micro': recall_micro,
            'precision_misinformation': precision_mis,
            'recall_misinformation': recall_mis,
            'f1_misinformation': f1_mis,
        }

        return metrics

Set BERT model

In [46]:
trainer = Trainer(
    model=model,                         
    args=train_args,             
    train_dataset=dataset["train"],        
    eval_dataset=dataset["test"],           
    compute_metrics=compute_metrics_standard     
)

Train BERT model 

In [ ]:
trainer.train()

Test performance for accurate information label

Load model

In [ ]:
model_path = ""
model = AutoModelForSequenceClassification.from_pretrained(model_path)

Inference

In [10]:
from transformers import pipeline

# documentation: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline
pipe_classifier = pipeline(
    "text-classification",
    model=model,  
    tokenizer=tokenizer,
    framework="pt"
)

WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu121 with CUDA 1201 (you have 2.4.0+cpu)
    Python  3.11.9 (you have 3.11.2)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


In [ ]:
text_lst = df_test["text"].tolist()

#inference
pipe_output = pipe_classifier(
    text_lst, 
    batch_size=64 
)
print(pipe_output)

df_output = pd.DataFrame(pipe_output)

# add inference data to original dataframe
df_test["label_text_pred"] = df_output["label"].tolist()
df_test["label_text_pred_probability"] = df_output["score"].round(2).tolist()
#Print df_test
df_test

Calculate f1, precision recall

In [17]:
# Convert 'label_text_pred' to binary values
df_test['label_pred'] = df_test['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)
precision = precision_recall_fscore_support(y_true=df_test['label'], y_pred=df_test['label_pred'], average='binary', pos_label =0)
print(f'Precision, recall, fscore: {precision}')

Precision, recall, fscore: (0.8776881720430108, 0.9546783625730995, 0.9145658263305323, None)
